## Импорт библиотек

In [9]:
import sys
import os
from tqdm import tqdm

import pandas as pd
import numpy as np

In [10]:
from thermodynamics import atmosphere_standard

In [3]:
class Inlet:
    """Входное устройство"""

    def get_inlet_parameters(self, **kwargs) -> None:
        """Расчет параметров перед"""
        # Невозмущенные параметры
        self.R_gas_i = self.substance.R()
        if not hasattr(self, 'T1') and self.substance.strip().upper() == 'AIR':
            self.T1 = atmosphere_standard(kwargs.get('H', None))['T']
        if not hasattr(self, 'P1') and self.substance.strip().upper() == 'AIR':
            self.P1 = atmosphere_standard(kwargs.get('H', None))['P']
        assert self.T1 and self.P1, f'{type(self).__name__} object has no attributes T1 and P1!'
        self.ρ1 = self.P1 / (self.R_gas1 * self.T1)
        self.g1 = 1
        self.Cp1 = Cp(self.substance, T=self.T1, P=self.P1, a_ox=getattr(self, 'a_ox1', None), fuel=fuel)
        self.k1 = self.Cp1 / (self.Cp1 - self.R_gas1)
        self.Mc1 = 0
        self.c1 = 0
        self.TT1 = self.T1
        self.PP1 = self.P1
        self.ρρ1 = self.ρ1

    def calculate(self, G, *args, **kwargs):
        """Расчет параметров перед"""

        T1 = 288
        self.T_o = T1
        P1 = 100_000
        P3 = P1
        return {'T_o': self.T_o, 'P3': P3}

In [4]:
class Compressor:

    def calculate(self, G, *args, **kwargs):
        Cp = 1004
        T1 = scheme[self.place['contour']][self.place['node']-1].T_o
        #T1 = 288
        T3 = 800
        Lc = Cp * (T3 - T1)
        return {'N': Lc * G, 'G': G * 0.999}

In [5]:
class CombustionChamber:

    def calculate(self, G, *args, **kwargs):
        a_ox3 = 1.2
        l0 = 14
        g_fuel = 1 / l0 / a_ox3

In [6]:
class Turbine:
    def calculate(self, G, ππ, *args, **kwargs):
        k = 1.33
        T1 = 1650
        T2 = 1300
        eff = 0.9
        Lt = -k / (k - 1) * 288 * T1 * (1 - 1 / (ππ ** ((k - 1) / k))) * eff
        return {'N': Lt * G, 'G': G}

In [7]:
class Nozzle:
    def calculate(self, G, *args, **kwargs):
        Cp2 = 1200
        TT3 = 600
        ππ = 1.4
        k2 = 1.33
        v_ = 0.99
        c3 = v_ * (2 * Cp2 * TT3 * (1 - ππ ** ((1 - k2) / k2))) ** 0.5
        self.R = c3 * G
        return {'R': self.R}

In [8]:
class GTE:
    """ГТД"""

    def __init__(self, name='GTE') -> None:
        self.name = name
        self.scheme = {}  # схема ГТД
        self.shafts = []  # валы ГТД
        self.contouring = {1: 1}  # степень контурности []

        self.M = nan  # Мах полета []
        self.v = nan  # скорость полета [м/с]

    def __str__(self) -> str:
        return self.name

    def eq(self, points, *args, **kwargs):
        res = []

        # уравнения неразрывности
        '''for i in range(1):
            res.append(Turbine(points[0], points[1])['G'] - Compressor.calculate(points[0])['G'])'''

        # Баланс мощностей
        for contour, shaft in self.shafts.items():
            res.append(sum([node.calculate(*points, *args, **kwargs)['N'] for i, node in enumerate(shaft)]))

        res.append(self.scheme[1][-1].calculate(points[0])['R'] - self.R)

        return res

    def get_varibles(self):
        # Массовй расход
        vars = {'G': 300}

        # Степени понижения полного давления в турбинах
        for contour in self.scheme:
            for i, node in enumerate(self.scheme[contour]):
                if isinstance(node, Turbine):
                    vars['pipi' + str(i)] = 3
        print(f'points0: {vars}')
        return vars

    @timeit()
    def calculate(self, *args, **kwargs):
        vars0 = self.get_varibles()
        vars_list = fsolve(self.eq, list(vars0.values()))
        vars = dict()
        for i, key in enumerate(vars0.keys()):
            vars[key] = vars_list[i]
        print(Fore.GREEN + f'points: {vars}')
        print(Fore.CYAN + f'eq: {gte.eq(list(vars.values()))}')
        return vars

NameError: name 'timeit' is not defined

In [ ]:
from gte import GTE

from inlet import Inlet
from compressor import Compressor
from combustion_chambler import CombustionChamber
from turbine import Turbine
from nozzle import Nozzle

Matplotlib is building the font cache; this may take a moment.


file "table_values/Атмосфера стандартная.xlsx" did not found!


FileNotFoundError: [Errno 2] No such file or directory: 'table_values/Теплоёмкость воздуха.xlsx'